# CyberPhi — Colab Quickstart

**Purpose:** Smoke-test the full pipeline (data → train → inference) on a free Colab GPU before committing to a RunPod instance.

**Before running:**
1. Go to `Runtime → Change runtime type` and select **GPU** (T4 is fine for testing)
2. Add your API keys to Colab Secrets (`🔑` icon in the left sidebar):
   - `ANTHROPIC_API_KEY`
   - `NVD_API_KEY` *(optional — without it NVD rate-limits to 5 req/30 s)*
   - `HUGGINGFACE_TOKEN` *(needed for Alpaca mixing step)*
3. Run cells top to bottom — each section is independent so you can re-run any step

**What this notebook does differently from a full RunPod run:**
- Data pipeline is capped at `DATA_LIMIT` rows (default 50) to keep API costs low
- Training is 1 epoch with a T4-safe config (sequence_len=2048, fp16, no flash-attn)
- Inference uses `transformers` + the LoRA adapter directly — no Ollama needed
- Everything writes to `/content/cyberphi/` (lost when session ends; mount Drive to persist)

---
## 0 — GPU Check

In [ ]:
import subprocess, textwrap

result = subprocess.run(['nvidia-smi', '--query-gpu=name,memory.total,driver_version',
                         '--format=csv,noheader'], capture_output=True, text=True)
if result.returncode != 0:
    raise RuntimeError('No GPU detected — go to Runtime → Change runtime type and select GPU')

gpu_name, vram, driver = [s.strip() for s in result.stdout.strip().split(',')]
print(f'GPU:    {gpu_name}')
print(f'VRAM:   {vram}')
print(f'Driver: {driver}')

# Determine profile: A100/L4 can use the full config; T4/P100/V100 need a lighter one
HIGH_END_GPUS = ('A100', 'A10', 'L4', 'H100')
IS_HIGH_END   = any(g in gpu_name for g in HIGH_END_GPUS)
print(f'\nProfile: {"high-end (full config)" if IS_HIGH_END else "T4/P100 (lightweight config)"}')

---
## 1 — Clone Repo

In [ ]:
# ── Optional: mount Drive so checkpoints survive session resets ──────────────
MOUNT_DRIVE = False   # set True if you want persistence

if MOUNT_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')
    REPO_DIR = '/content/drive/MyDrive/cyberphi'
else:
    REPO_DIR = '/content/cyberphi'

import os
if not os.path.exists(REPO_DIR):
    # Replace with your actual repo URL
    REPO_URL = 'https://github.com/boyatoid/CyberPhi.git'
    !git clone {REPO_URL} {REPO_DIR}
else:
    print(f'Repo already exists at {REPO_DIR} — pulling latest')
    !git -C {REPO_DIR} pull

%cd {REPO_DIR}
print(f'Working directory: {os.getcwd()}')

---
## 2 — API Keys

In [ ]:
from google.colab import userdata
import os

def _get_secret(name, required=True):
    try:
        val = userdata.get(name)
        if val:
            return val
    except Exception:
        pass
    if required:
        raise RuntimeError(f'Secret "{name}" not found — add it in the 🔑 Secrets panel')
    return ''

ANTHROPIC_API_KEY = _get_secret('ANTHROPIC_API_KEY')
NVD_API_KEY       = _get_secret('NVD_API_KEY',      required=False)
HF_TOKEN          = _get_secret('HUGGINGFACE_TOKEN', required=False)

# Write .env so dataset/config.py picks them up
with open('.env', 'w') as f:
    f.write(f'ANTHROPIC_API_KEY={ANTHROPIC_API_KEY}\n')
    if NVD_API_KEY:
        f.write(f'NVD_API_KEY={NVD_API_KEY}\n')
    if HF_TOKEN:
        f.write(f'HUGGINGFACE_TOKEN={HF_TOKEN}\n')

# Also export for child processes
os.environ['ANTHROPIC_API_KEY'] = ANTHROPIC_API_KEY
if NVD_API_KEY:
    os.environ['NVD_API_KEY'] = NVD_API_KEY
if HF_TOKEN:
    os.environ['HUGGINGFACE_TOKEN'] = HF_TOKEN

print('ANTHROPIC_API_KEY: set ✓')
print(f'NVD_API_KEY:       {"set ✓" if NVD_API_KEY else "not set (rate-limited to 5 req/30s)"}')
print(f'HUGGINGFACE_TOKEN: {"set ✓" if HF_TOKEN else "not set (Alpaca mixing will be skipped)"}')

---
## 3 — Install Dependencies

In [ ]:
# Dataset pipeline deps
!pip install -r requirements.txt -q
print('requirements.txt installed ✓')

In [ ]:
# Axolotl — skip flash-attn on T4 (requires Ampere+); install it on high-end GPUs
if IS_HIGH_END:
    !pip install axolotl[flash-attn] -q
else:
    !pip install axolotl -q

import axolotl
print(f'Axolotl {axolotl.__version__} installed ✓')

---
## 4 — Data Pipeline (limited)

Set `DATA_LIMIT` to control how many rows each scraper fetches.  
50 rows ≈ a few minutes and < $0.10 in Claude API cost — enough to validate the pipeline.

In [ ]:
DATA_LIMIT = 50   # rows per scraper — increase for real data builds

!python dataset/pipeline.py all --limit {DATA_LIMIT}

# Verify output
import json
from pathlib import Path

validated = Path('data/final/validated.jsonl')
if not validated.exists() or validated.stat().st_size == 0:
    raise RuntimeError('Pipeline produced no output — check logs above')

rows = validated.read_text().strip().splitlines()
print(f'\n✓ {len(rows)} validated training rows')

# Show one sample
sample = json.loads(rows[0])
print(f'\nSample entry:')
print(f'  source:     {sample["source"]}')
print(f'  vuln_type:  {sample["vuln_type"]}')
print(f'  severity:   {sample["severity"]}')
print(f'  instruction:{sample["instruction"][:80]}…')

---
## 5 — Training

The config is patched automatically for the detected GPU:  
- **T4/P100**: `sequence_len=2048`, `fp16=True`, `flash_attention=False`, `num_epochs=1`  
- **A100/L4**: original config values, `num_epochs=1` (still capped for Colab)

In [ ]:
import yaml, copy
from pathlib import Path

RUN_NAME   = 'colab_test'
OUTPUT_DIR = f'outputs/{RUN_NAME}'
COLAB_CFG  = 'training/axolotl_config_colab.yaml'

# Load base config
with open('training/axolotl_config.yaml') as f:
    cfg = yaml.safe_load(f)

# Always override for Colab (even A100 sessions disconnect, so 1 epoch)
cfg['output_dir'] = OUTPUT_DIR
cfg['num_epochs'] = 1
cfg['save_steps'] = 50
cfg['eval_steps'] = 50

if not IS_HIGH_END:
    # T4 / P100 safe settings
    cfg['sequence_len']               = 2048
    cfg['micro_batch_size']           = 1
    cfg['gradient_accumulation_steps']= 8     # effective batch = 8
    cfg['flash_attention']            = False
    cfg['bf16']                       = False
    cfg['fp16']                       = True
    print('Applied T4-safe overrides: sequence_len=2048, fp16, no flash-attn')
else:
    print('High-end GPU: using original config values (flash-attn, bf16)')

Path(COLAB_CFG).parent.mkdir(parents=True, exist_ok=True)
with open(COLAB_CFG, 'w') as f:
    yaml.dump(cfg, f, default_flow_style=False, allow_unicode=True)

print(f'Colab config written → {COLAB_CFG}')
print(f'Output dir           → {OUTPUT_DIR}')

In [ ]:
!axolotl train {COLAB_CFG}

In [ ]:
# Verify adapter was saved
adapter_files = list(Path(OUTPUT_DIR).rglob('adapter_model*'))
if not adapter_files:
    print('WARNING: no adapter_model files found — training may have failed')
else:
    print(f'✓ Adapter saved:')
    for f in adapter_files:
        size_mb = f.stat().st_size / 1e6
        print(f'  {f}  ({size_mb:.1f} MB)')

---
## 6 — Inference Test

Loads the base model + LoRA adapter directly with `transformers` and `peft`  
(no Ollama needed on Colab). This validates the adapter actually works.

In [ ]:
!pip install peft transformers accelerate bitsandbytes -q
print('Inference deps installed ✓')

In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from peft import PeftModel

BASE_MODEL = 'microsoft/Phi-3.5-mini-instruct'

print('Loading tokenizer …')
tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL, trust_remote_code=True)

print('Loading base model in 4-bit …')
bnb_cfg = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_quant_type='nf4',
    bnb_4bit_use_double_quant=True,
)
base = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL,
    quantization_config=bnb_cfg,
    device_map='auto',
    trust_remote_code=True,
)

print(f'Loading LoRA adapter from {OUTPUT_DIR} …')
model = PeftModel.from_pretrained(base, OUTPUT_DIR)
model.eval()
print('Model ready ✓')

In [ ]:
SYSTEM = 'You are a senior cybersecurity expert and penetration tester.'
QUERY  = 'Explain how SQL injection works and give a simple example of a vulnerable PHP snippet.'

# ChatML format (matches training)
prompt = (
    f'<|im_start|>system\n{SYSTEM}<|im_end|>\n'
    f'<|im_start|>user\n{QUERY}<|im_end|>\n'
    f'<|im_start|>assistant\n'
)

inputs = tokenizer(prompt, return_tensors='pt').to(model.device)

print(f'Query: {QUERY}\n')
print('--- Response ---')
with torch.no_grad():
    out = model.generate(
        **inputs,
        max_new_tokens=512,
        temperature=0.7,
        do_sample=True,
        pad_token_id=tokenizer.eos_token_id,
        eos_token_id=tokenizer.convert_tokens_to_ids('<|im_end|>'),
    )

response = tokenizer.decode(out[0][inputs['input_ids'].shape[1]:], skip_special_tokens=True)
print(response)

---
## 7 — Evaluation Metrics (no Ollama needed)

Runs the metrics that don't require Ollama — ROUGE-L, think-block rate, CoT step distribution.

In [ ]:
import json, sys
sys.path.insert(0, '.')
from evaluation.metrics import (
    rouge_l_score, think_block_presence_rate,
    cot_step_count_distribution, vuln_class_accuracy,
)
from pathlib import Path

# Run inference on the first N rows of the validated set and score
EVAL_LIMIT = 5
rows = Path('data/final/validated.jsonl').read_text().strip().splitlines()[:EVAL_LIMIT]
samples = [json.loads(r) for r in rows]

outputs  = []
refs     = []
classes  = []

for s in samples:
    prompt = (
        f'<|im_start|>system\n{SYSTEM}<|im_end|>\n'
        f'<|im_start|>user\n{s["instruction"]}\n{s["input"]}<|im_end|>\n'
        f'<|im_start|>assistant\n'
    )
    inputs = tokenizer(prompt, return_tensors='pt').to(model.device)
    with torch.no_grad():
        out = model.generate(
            **inputs, max_new_tokens=256, do_sample=False,
            pad_token_id=tokenizer.eos_token_id,
            eos_token_id=tokenizer.convert_tokens_to_ids('<|im_end|>'),
        )
    response = tokenizer.decode(out[0][inputs['input_ids'].shape[1]:], skip_special_tokens=True)
    outputs.append(response)
    refs.append(s['output'])
    classes.append(s.get('vuln_type', ''))
    print(f'  [{s["source"]}] {s["instruction"][:60]}…  →  {len(response)} chars')

rouge_scores = [rouge_l_score(o, r) for o, r in zip(outputs, refs)]
print(f'\n=== Eval over {len(samples)} samples ===')
print(f'Mean ROUGE-L:          {sum(rouge_scores)/len(rouge_scores):.3f}')
print(f'<think> block rate:    {think_block_presence_rate(outputs):.1%}')
print(f'Vuln-class accuracy:   {vuln_class_accuracy(outputs, classes):.1%}')
print(f'CoT step distribution: {cot_step_count_distribution(outputs)}')

---
## 8 — (Optional) Download Adapter

Download the LoRA adapter to your machine so you can upload it to RunPod for a full training run.

In [ ]:
import shutil
from google.colab import files
from pathlib import Path

zip_path = f'/content/{RUN_NAME}_adapter.zip'
shutil.make_archive(zip_path.replace('.zip', ''), 'zip', OUTPUT_DIR)
files.download(zip_path)
print(f'Downloaded {zip_path}')